# 06 - WC2026 Results Presentation

Ce notebook presente les resultats de simulation (qualif, phases finales, vainqueur) avec des visualisations claires.

In [1]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "vscode"

DATA_PATH = Path("../data/processed/wc2026_simulation.csv")
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
df.head()

,team,qual_prob,r16_prob,qf_prob,sf_prob,final_prob,win_prob
0,Argentina,0.957,0.749,0.534,0.347,0.226,0.128
1,France,0.937,0.682,0.472,0.316,0.190,0.113
2,Brazil,0.933,0.700,0.471,0.297,0.184,0.112
3,Belgium,0.974,0.763,0.531,0.342,0.204,0.105
4,England,0.878,0.632,0.388,0.221,0.134,0.080


## Resume rapide

In [2]:
summary = df[["qual_prob", "r16_prob", "qf_prob", "sf_prob", "final_prob", "win_prob"]].describe()
summary

,qual_prob,r16_prob,qf_prob,sf_prob,final_prob,win_prob
count,48.000000,48.000000,48.000000,48.000000,48.000000,48.000000
mean,0.666667,0.333333,0.166667,0.083333,0.041667,0.020833
std,0.180763,0.204705,0.151464,0.101006,0.061658,0.035055
min,0.357000,0.088000,0.021000,0.005000,0.001000,0.000000
25%,0.503750,0.158250,0.051750,0.014000,0.004000,0.001000
50%,0.654500,0.282500,0.100000,0.034000,0.013000,0.004500
75%,0.823750,0.466500,0.224250,0.105500,0.040750,0.018000
max,0.974000,0.763000,0.534000,0.347000,0.226000,0.128000


In [3]:
import numpy as np

# On charge les données
df = pd.read_csv(DATA_PATH)

# Normalisation rapide pour la visualisation (on ramène le max à 100%)
cols_to_norm = ["qual_prob", "r16_prob", "qf_prob", "sf_prob", "final_prob", "win_prob"]
for col in cols_to_norm:
    if df[col].max() > 1:
        df[col] = df[col] / df[col].max()

# On trie par probabilité de victoire
df_top = df.sort_values("win_prob", ascending=False).head(15)

In [4]:
fig_win = px.bar(
    df_top.head(10),
    x="win_prob",
    y="team",
    orientation='h',
    title="<b>Top 10 - Probabilité de Victoire Finale (WC 2026)</b>",
    labels={"win_prob": "Probabilité de titre", "team": "Équipe"},
    color="win_prob",
    color_continuous_scale="Viridis",
    text_auto='.1%'
)
fig_win.update_layout(yaxis={'categoryorder':'total ascending'}, showlegend=False)
fig_win.show()

In [5]:
# Préparation des données pour la heatmap
heatmap_data = df_top.set_index("team")[cols_to_norm]
heatmap_data.columns = ["Poules", "8èmes", "Quarts", "Demis", "Finale", "Vainqueur"]

fig_heat = px.imshow(
    heatmap_data,
    labels=dict(x="Stade de la compétition", y="Équipe", color="Probabilité"),
    x=heatmap_data.columns,
    y=heatmap_data.index,
    color_continuous_scale="RdYlGn",
    title="<b>Chances de survie par étape</b>",
    aspect="auto"
)
fig_heat.update_xaxes(side="top")
fig_heat.show()

In [6]:
fig_scatter = px.scatter(
    df_top,
    x="qual_prob",
    y="win_prob",
    text="team",
    size="win_prob",
    color="win_prob",
    title="<b>Stabilité en poules vs Capacité à conclure</b>",
    labels={"qual_prob": "Chances de sortir des poules", "win_prob": "Chances de titre"},
    hover_name="team"
)
fig_scatter.update_traces(textposition='top center')
fig_scatter.show()

In [7]:
fig_map = px.choropleth(
    df,
    locations="team",
    locationmode="country names",
    color="win_prob",
    hover_name="team",
    color_continuous_scale="OrRd",
    title="<b>Géographie des Favoris - WC 2026</b>",
    labels={'win_prob': 'Chances de titre'}
)
fig_map.show()

/var/folders/_j/t0knsx_d5_z7c38r3jzmx7rm0000gn/T/ipykernel_9476/3617330382.py:1: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig_map = px.choropleth(


In [8]:
import joblib
import pandas as pd
import plotly.express as px
import numpy as np

# 1. Chargement du modèle
model = joblib.load("../data/processed/models/champion_xgboost.pkl")

# 2. Récupérer le premier modèle et son pipeline de transformation
first_fold = model.calibrated_classifiers_[0].estimator
preprocessor = first_fold.steps[0][1]  # Le transformer (ColumnTransformer / Encoder)
xgboost_model = first_fold.steps[-1][1] # Le modèle XGBoost

# 3. Tenter de récupérer les noms de colonnes générés
try:
    # Si ton pipeline supporte get_feature_names_out (Scikit-learn moderne)
    feature_names_transformed = preprocessor.get_feature_names_out()
    print(f"✅ {len(feature_names_transformed)} noms de colonnes récupérés du preprocessor.")
except:
    print("⚠️ Impossible de récupérer les noms via get_feature_names_out.")
    # Fallback : on crée des noms génériques pour ne pas bloquer
    feature_names_transformed = [f"feature_{i}" for i in range(len(xgboost_model.feature_importances_))]

# 4. Extraction de l'importance
importances = xgboost_model.feature_importances_

# 5. Création du DataFrame (en s'assurant que les longueurs matchent)
min_len = min(len(feature_names_transformed), len(importances))
df_imp = pd.DataFrame({
    'Feature': feature_names_transformed[:min_len],
    'Importance': importances[:min_len]
}).sort_values('Importance', ascending=False)

print(df_imp.head(20))
# 6. Plot
fig = px.bar(
    df_imp.head(20), 
    x='Importance', 
    y='Feature', 
    orientation='h',
    title="<b>Top 20 : Ce que ton modèle regarde vraiment</b>",
    color='Importance',
    color_continuous_scale='Magma'
)
fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.show()

✅ 79 noms de colonnes récupérés du preprocessor.
                                  Feature  Importance
66  num__away_squad_per_90_minutes_g_a_pk    0.039168
21                  num__fifa_points_diff    0.026392
30       num__home_squad_playing_time_min    0.021789
76       cat__away_confederation_CONMEBOL    0.020347
44  num__home_squad_per_90_minutes_g_a_pk    0.019464
65    num__away_squad_per_90_minutes_g_pk    0.018330
57       num__away_squad_performance_g_pk    0.018220
20                    num__fifa_rank_diff    0.017217
40     num__home_squad_per_90_minutes_gls    0.016894
49                   num__away_squad_born    0.015939
77            cat__away_confederation_OFC    0.015875
56        num__away_squad_performance_g_a    0.015248
38       num__home_squad_performance_crdy    0.014876
53       num__away_squad_playing_time_90s    0.014466
5                           num__elo_diff    0.014305
64     num__away_squad_per_90_minutes_g_a    0.014277
24                   num__home_sq